# 3. Build and validate the KIGB sample

This notebook reintroduces the intermediate population, defines the KIGB target, refits the reduced scorecard, checks calibration around the proposed policy boundaries and compares operational cut-offs on development and chronological holdout data.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from sklearn.metrics import brier_score_loss, roc_auc_score

project_root = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data" / "KGB Modelling" / "KGB in sample.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate KGB modelling data")

kgb_dir = project_root / "data" / "KGB Modelling"
kigb_dir = project_root / "data" / "KIGB Modelling"
kigb_dir.mkdir(parents=True, exist_ok=True)
known_train = pd.read_csv(kgb_dir / "KGB in sample.csv")
known_test = pd.read_csv(kgb_dir / "KGB out sample.csv")
intermediate = pd.read_csv(kgb_dir / "indeterminates.csv")
split_date = pd.Timestamp("2024-07-01")

for rows in [known_train, known_test, intermediate]:
    rows["application_date"] = pd.to_datetime(rows["application_date"], errors="raise")

intermediate_train = intermediate.loc[intermediate["application_date"] < split_date].copy()
intermediate_test = intermediate.loc[intermediate["application_date"] >= split_date].copy()

def add_source_group(data, label):
    result = data.copy()
    result["source_group"] = label
    return result

kigb_train = pd.concat([
    add_source_group(known_train, "Known good/bad"),
    add_source_group(intermediate_train, "Intermediate"),
], ignore_index=True)
kigb_test = pd.concat([
    add_source_group(known_test, "Known good/bad"),
    add_source_group(intermediate_test, "Intermediate"),
], ignore_index=True)

for rows in [kigb_train, kigb_test]:
    missed = pd.to_numeric(rows["missed_payment_months"], errors="raise")
    rows["kigb_bad"] = missed.ge(3).astype(int)
    rows["outcome_group"] = np.select(
        [missed.eq(0), missed.between(1, 2), missed.ge(3)],
        ["Known good", "Intermediate", "Known bad"],
        default="Invalid",
    )
    if rows["outcome_group"].eq("Invalid").any():
        raise ValueError("Invalid missed-payment count")

population_summary = pd.DataFrame({
    "Development": kigb_train["outcome_group"].value_counts(),
    "Chronological holdout": kigb_test["outcome_group"].value_counts(),
}).fillna(0).astype(int)
display(population_summary)
print(f"Development dates: {kigb_train.application_date.min().date()} to {kigb_train.application_date.max().date()}")
print(f"Holdout dates: {kigb_test.application_date.min().date()} to {kigb_test.application_date.max().date()}")

,Development,Chronological holdout
outcome_group,,
Known good,6964,2356
Known bad,294,102
Intermediate,147,50


Development dates: 2023-01-01 to 2024-06-30
Holdout dates: 2024-07-01 to 2024-12-30


## Fit the reduced specification on KIGB development data

In [2]:
scaled_predictors = [
    "debt_to_income_ratio", "recent_application_count", "revolving_utilisation",
    "historic_defaults", "log_address_tenure",
]
numeric_inputs = [
    "debt_to_income_ratio", "recent_application_count", "revolving_utilisation",
    "historic_defaults", "disposable_income", "time_at_address_months",
]

def prepare_inputs(data, target):
    rows = data.copy()
    for column in numeric_inputs:
        rows[column] = pd.to_numeric(rows[column], errors="raise")
    required = [target, "loan_purpose", *numeric_inputs]
    if rows[required].isna().any().any():
        raise ValueError("Missing model input or target")
    if not np.isfinite(rows[numeric_inputs].to_numpy()).all():
        raise ValueError("Infinite model input")
    if rows["time_at_address_months"].lt(0).any():
        raise ValueError("Negative address tenure")
    rows["low_disposable_income_flag"] = (rows["disposable_income"] < 150).astype(float)
    rows["log_address_tenure"] = np.log1p(rows["time_at_address_months"])
    return rows

def make_design(rows, means, stds):
    x = (rows[scaled_predictors] - means) / stds
    x.insert(0, "const", 1.0)
    x["low_disposable_income_flag"] = rows["low_disposable_income_flag"]
    x["loan_purpose=education"] = (rows["loan_purpose"] == "education").astype(float)
    return x.astype(float)

factor = 20 / np.log(2)
offset = 600 - factor * np.log(50)

def pd_to_score(probability):
    p = np.clip(np.asarray(probability, dtype=float), 1e-12, 1 - 1e-12)
    return offset + factor * np.log((1 - p) / p)

kigb_train = prepare_inputs(kigb_train, "kigb_bad")
kigb_test = prepare_inputs(kigb_test, "kigb_bad")
means = kigb_train[scaled_predictors].mean()
stds = kigb_train[scaled_predictors].std(ddof=0)
x_train = make_design(kigb_train, means, stds)
x_test = make_design(kigb_test, means, stds)[x_train.columns]
y_train = kigb_train["kigb_bad"].astype(int)
y_test = kigb_test["kigb_bad"].astype(int)

kigb_model = sm.Logit(y_train, x_train).fit(maxiter=200, disp=False)
if not kigb_model.mle_retvals.get("converged", False):
    raise RuntimeError("KIGB model did not converge")

kigb_train["predicted_pd"] = np.asarray(kigb_model.predict(x_train))
kigb_test["predicted_pd"] = np.asarray(kigb_model.predict(x_test))
kigb_train["score"] = pd_to_score(kigb_train["predicted_pd"])
kigb_test["score"] = pd_to_score(kigb_test["predicted_pd"])

def model_metrics(name, rows):
    auc = roc_auc_score(rows["kigb_bad"], rows["predicted_pd"])
    return {
        "Sample": name, "Accounts": len(rows),
        "Observed bad rate": rows["kigb_bad"].mean(),
        "Mean predicted PD": rows["predicted_pd"].mean(),
        "Calibration gap": rows["kigb_bad"].mean() - rows["predicted_pd"].mean(),
        "Brier score": brier_score_loss(rows["kigb_bad"], rows["predicted_pd"]),
        "ROC AUC": auc, "Gini": 2 * auc - 1,
    }

overall = pd.DataFrame([
    model_metrics("Development", kigb_train),
    model_metrics("Chronological holdout", kigb_test),
])
display(overall.style.format({
    "Accounts": "{:,.0f}", "Observed bad rate": "{:.2%}", "Mean predicted PD": "{:.2%}",
    "Calibration gap": "{:+.2%}", "Brier score": "{:.4f}", "ROC AUC": "{:.4f}", "Gini": "{:.4f}",
}))

,Sample,Accounts,Observed bad rate,Mean predicted PD,Calibration gap,Brier score,ROC AUC,Gini
0,Development,"7,405",3.97%,3.97%,+0.00%,0.0378,0.6385,0.2769
1,Chronological holdout,"2,508",4.07%,3.83%,+0.24%,0.0386,0.6429,0.2858


## Calibration around the proposed 2.50% and 3.50% boundaries

In [3]:
band_edges = [0, 0.015, 0.020, 0.025, 0.030, 0.035, 0.050, 0.075, 0.10, 1.000001]
labels = [
    "<1.50%", "1.50–1.99%", "2.00–2.49%", "2.50–2.99%", "3.00–3.49%",
    "3.50–4.99%", "5.00–7.49%", "7.50–9.99%", "10.00%+",
]

def calibration_by_band(sample_name, rows):
    grouped = rows.copy()
    grouped["PD band"] = pd.cut(
        grouped["predicted_pd"], band_edges, labels=labels,
        include_lowest=True, right=False,
    )
    table = grouped.groupby("PD band", observed=False).agg(
        Accounts=("kigb_bad", "size"),
        Observed_bads=("kigb_bad", "sum"),
        Mean_predicted_PD=("predicted_pd", "mean"),
        Observed_bad_rate=("kigb_bad", "mean"),
        Mean_score=("score", "mean"),
    ).reset_index()
    table.insert(0, "Sample", sample_name)
    table["Calibration gap"] = table["Observed_bad_rate"] - table["Mean_predicted_PD"]
    return table

band_calibration = pd.concat([
    calibration_by_band("Development", kigb_train),
    calibration_by_band("Chronological holdout", kigb_test),
], ignore_index=True)
display(band_calibration.style.format({
    "Accounts": "{:,.0f}", "Observed_bads": "{:,.0f}", "Mean_predicted_PD": "{:.2%}",
    "Observed_bad_rate": "{:.2%}", "Calibration gap": "{:+.2%}", "Mean_score": "{:.1f}",
}, na_rep="—"))

,Sample,PD band,Accounts,Observed_bads,Mean_predicted_PD,Observed_bad_rate,Mean_score,Calibration gap
0,Development,<1.50%,212,2,1.09%,0.94%,617.6,-0.15%
1,Development,1.50–1.99%,226,4,1.83%,1.77%,602.2,-0.06%
2,Development,2.00–2.49%,834,22,2.28%,2.64%,595.6,+0.35%
3,Development,2.50–2.99%,"1,271",27,2.75%,2.12%,590.1,-0.62%
4,Development,3.00–3.49%,"1,225",34,3.24%,2.78%,585.1,-0.47%
5,Development,3.50–4.99%,"2,099",94,4.12%,4.48%,578.1,+0.36%
6,Development,5.00–7.49%,"1,102",69,5.96%,6.26%,566.9,+0.30%
7,Development,7.50–9.99%,313,31,8.46%,9.90%,555.9,+1.45%
8,Development,10.00%+,123,11,12.36%,8.94%,544.2,-3.42%
9,Chronological holdout,<1.50%,87,1,1.09%,1.15%,617.8,+0.06%


## Cut-off comparison

The predicted accepted-book bad rate is the mean PD among accepted applicants. The observed accepted-book bad rate is the realised proportion with `kigb_bad = 1`. The 2% target is applied to the predicted measure; observed performance is reported separately and is the stronger validation check.

In [4]:
cutoffs = [0.0200, 0.0225, 0.0250, 0.0300, 0.0350]

def cutoff_row(sample_name, rows, cutoff):
    accepted = rows.loc[rows["predicted_pd"] < cutoff]
    flagged = rows.loc[rows["predicted_pd"] >= cutoff]
    intermediate_mask = rows["outcome_group"].eq("Intermediate")
    return {
        "Sample": sample_name, "PD cutoff": cutoff,
        "Minimum score": float(pd_to_score([cutoff])[0]),
        "Accepted accounts": len(accepted), "Approval rate": len(accepted) / len(rows),
        "Predicted accepted bad rate": accepted["predicted_pd"].mean(),
        "Observed accepted bad rate": accepted["kigb_bad"].mean(),
        "Accepted calibration gap": accepted["kigb_bad"].mean() - accepted["predicted_pd"].mean(),
        "Bad capture in flagged group": flagged["kigb_bad"].sum() / rows["kigb_bad"].sum(),
        "Intermediate accepted": int((intermediate_mask & rows["predicted_pd"].lt(cutoff)).sum()),
        "Intermediate flagged": int((intermediate_mask & rows["predicted_pd"].ge(cutoff)).sum()),
    }

cutoff_results = pd.DataFrame([
    cutoff_row(sample_name, rows, cutoff)
    for sample_name, rows in [("Development", kigb_train), ("Chronological holdout", kigb_test)]
    for cutoff in cutoffs
])
display(cutoff_results.style.format({
    "PD cutoff": "{:.2%}", "Minimum score": "{:.1f}", "Approval rate": "{:.1%}",
    "Predicted accepted bad rate": "{:.2%}", "Observed accepted bad rate": "{:.2%}",
    "Accepted calibration gap": "{:+.2%}", "Bad capture in flagged group": "{:.1%}",
}))

,Sample,PD cutoff,Minimum score,Accepted accounts,Approval rate,Predicted accepted bad rate,Observed accepted bad rate,Accepted calibration gap,Bad capture in flagged group,Intermediate accepted,Intermediate flagged
0,Development,2.00%,599.4,438,5.9%,1.47%,1.37%,-0.10%,98.0%,3,144
1,Development,2.25%,595.9,768,10.4%,1.76%,1.69%,-0.06%,95.6%,7,140
2,Development,2.50%,592.8,1272,17.2%,2.00%,2.20%,+0.20%,90.5%,12,135
3,Development,3.00%,587.4,2543,34.3%,2.37%,2.16%,-0.21%,81.3%,33,114
4,Development,3.50%,582.8,3768,50.9%,2.66%,2.36%,-0.29%,69.7%,53,94
5,Chronological holdout,2.00%,599.4,158,6.3%,1.42%,1.90%,+0.48%,97.1%,2,48
6,Chronological holdout,2.25%,595.9,282,11.2%,1.73%,1.42%,-0.31%,96.1%,4,46
7,Chronological holdout,2.50%,592.8,459,18.3%,1.98%,1.53%,-0.46%,93.1%,8,42
8,Chronological holdout,3.00%,587.4,889,35.4%,2.36%,2.25%,-0.11%,80.4%,14,36
9,Chronological holdout,3.50%,582.8,1323,52.8%,2.65%,2.57%,-0.08%,66.7%,25,25


## Save scored KIGB samples for later validation

In [5]:
in_path = kigb_dir / "KIGB in sample.csv"
out_path = kigb_dir / "KIGB out sample.csv"
kigb_train["application_date"] = pd.to_datetime(kigb_train["application_date"]).dt.strftime("%Y-%m-%d")
kigb_test["application_date"] = pd.to_datetime(kigb_test["application_date"]).dt.strftime("%Y-%m-%d")
kigb_train.to_csv(in_path, index=False, lineterminator="\n")
kigb_test.to_csv(out_path, index=False, lineterminator="\n")
print(f"Saved {len(kigb_train):,} development rows to {in_path}")
print(f"Saved {len(kigb_test):,} holdout rows to {out_path}")

Saved 7,405 development rows to C:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\KIGB Modelling\KIGB in sample.csv
Saved 2,508 holdout rows to C:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\KIGB Modelling\KIGB out sample.csv
